# KG1 V198 micro-distillation Colab Pro run

Short continuation after V195. It trains from the best adapter found in Drive and never submits to Kaggle automatically.


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import importlib.util, os, pathlib, shutil, subprocess, sys, urllib.request, zipfile, hashlib
ROOT = pathlib.Path('/content/kg1_v198')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V198')
PACK = DRIVE_ROOT / 'kg1_v198_colab_pack.zip'
PACK_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/49c2c2fd118d1eba64ababde29c0218e57fa408b/runs/v198_micro_distill_colab_pack_20260503/kg1_v198_colab_pack.zip'
PACK_SHA256 = '7e3e41b55bb6f5736c3d5325c7b481f3b52ac918eb13c311e9a343f43f6dedca'
OUT = DRIVE_ROOT / 'output_v198'
BASELINE_DIR = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/init_adapter/final')
V195_OUT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/output_v195')
BASE_ADAPTER_MODEL_SHA256 = '3d16ba908a5c8808624f1abd8fdc2b29f92723f5c874761161c894d7e5759f21'
BASE_ADAPTER_CONFIG_SHA256 = 'e5499f128fde60d32d0595d427e4fe84d8abe6dbde1d80886c970e8184e4b743'

def sha256_path(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def adapter_ready(path):
    cfg = path / 'adapter_config.json'
    model = path / 'adapter_model.safetensors'
    if not cfg.exists() or not model.exists():
        return False
    if cfg.stat().st_size < 100 or model.stat().st_size < 1024:
        return False
    try:
        import json
        json.loads(cfg.read_text(encoding='utf-8'))
    except Exception:
        return False
    return True

def ensure_baseline_adapter():
    BASELINE_DIR.mkdir(parents=True, exist_ok=True)
    if adapter_ready(BASELINE_DIR):
        cfg_ok = sha256_path(BASELINE_DIR / 'adapter_config.json') == BASE_ADAPTER_CONFIG_SHA256
        model_ok = sha256_path(BASELINE_DIR / 'adapter_model.safetensors') == BASE_ADAPTER_MODEL_SHA256
        if cfg_ok and model_ok:
            return BASELINE_DIR
        print('Existing baseline adapter has SHA mismatch; deleting and redownloading fallback.')
        for p in [BASELINE_DIR / 'adapter_config.json', BASELINE_DIR / 'adapter_model.safetensors']:
            if p.exists():
                p.unlink()
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    print('Downloading public 0.86 baseline adapter to Drive fallback...')
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_config.json', output_dir=str(BASELINE_DIR), force_download=True)
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_model.safetensors', output_dir=str(BASELINE_DIR), force_download=True)
    assert adapter_ready(BASELINE_DIR), f'Missing baseline adapter files in {BASELINE_DIR}'
    assert sha256_path(BASELINE_DIR / 'adapter_config.json') == BASE_ADAPTER_CONFIG_SHA256
    assert sha256_path(BASELINE_DIR / 'adapter_model.safetensors') == BASE_ADAPTER_MODEL_SHA256
    return BASELINE_DIR

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if PACK.exists() and sha256_path(PACK) != PACK_SHA256:
    print('Existing Drive pack SHA mismatch; deleting stale pack and downloading the verified one.')
    PACK.unlink()
if not PACK.exists():
    print('Pack not found in Drive; trying GitHub URL...')
    try:
        urllib.request.urlretrieve(PACK_URL, PACK)
    except Exception as exc:
        raise RuntimeError(f'Pack missing. Upload kg1_v198_colab_pack.zip to {PACK} or push the branch so PACK_URL is valid: {PACK_URL}') from exc
pack_hash = sha256_path(PACK)
print('Pack SHA256:', pack_hash)
assert pack_hash == PACK_SHA256, f'Pack SHA mismatch: {pack_hash}'
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as zf:
    zf.extractall(ROOT)
assert (ROOT / 'data/v198/v198_micro_train.strict.jsonl').exists()
assert (ROOT / 'data/v198/v198_micro_val.strict.jsonl').exists()
assert (ROOT / 'scripts/hf_job_train_v90.py').exists()

candidates = [
    V195_OUT / 'final_adapter',
    V195_OUT / 'checkpoint-110',
    V195_OUT / 'checkpoint-75',
    V195_OUT / 'checkpoint-55',
]
INIT_ADAPTER = next((p for p in candidates if adapter_ready(p)), None)
if INIT_ADAPTER is None:
    print('No V195 adapter/checkpoint found; falling back to 0.86 baseline adapter.')
    INIT_ADAPTER = ensure_baseline_adapter()
print('INIT_ADAPTER =', INIT_ADAPTER)
print('Pack extracted to', ROOT)


Pack SHA256: 7e3e41b55bb6f5736c3d5325c7b481f3b52ac918eb13c311e9a343f43f6dedca
INIT_ADAPTER = /content/drive/MyDrive/KG1_NVIDIA_V195/output_v195/checkpoint-55
Pack extracted to /content/kg1_v198


In [8]:
%cd /content/kg1_v198
import importlib.util, os, subprocess, sys
os.environ.setdefault('MAX_JOBS', '4')
os.environ.setdefault('PIP_ROOT_USER_ACTION', 'ignore')

def pip_install(args):
    print('+ pip install', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def pip_uninstall(package_name):
    print('+ pip uninstall -y', package_name)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', package_name], check=False)

def install_if_missing(module_name, args):
    if importlib.util.find_spec(module_name) is None:
        pip_install(args)
    else:
        print(f'{module_name} already installed')

pip_uninstall('torchao')
pip_install(['--upgrade', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja==1.13.0'])
pip_install(['transformers==5.7.0', 'accelerate==1.13.0', 'peft==0.19.1', 'datasets==4.8.5', 'safetensors==0.7.0', 'huggingface_hub==1.13.0', 'sentencepiece==0.2.1', 'protobuf==7.34.1'])
install_if_missing('causal_conv1d', ['causal-conv1d==1.6.1', '--no-build-isolation'])
install_if_missing('mamba_ssm', ['mamba-ssm==2.3.1', '--no-build-isolation'])
assert importlib.util.find_spec('torchao') is None, 'torchao still installed; restart runtime and rerun cells from top'
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
print('mamba_ssm OK:', getattr(mamba_ssm, '__version__', 'unknown'))


/content/kg1_v198
+ pip uninstall -y torchao
+ pip install --upgrade pip setuptools wheel packaging ninja==1.13.0
+ pip install transformers==5.7.0 accelerate==1.13.0 peft==0.19.1 datasets==4.8.5 safetensors==0.7.0 huggingface_hub==1.13.0 sentencepiece==0.2.1 protobuf==7.34.1
causal_conv1d already installed
mamba_ssm already installed
mamba_ssm OK: 2.3.1


In [10]:
import pathlib, urllib.request

FIXED_TRAIN_SCRIPT_URL = "https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/31d439bc4a9b33b7b3c772d3526149847103a9b1/scripts/hf_job_train_v90.py"
TRAIN_SCRIPT = pathlib.Path("/content/kg1_v198/scripts/hf_job_train_v90.py")

urllib.request.urlretrieve(FIXED_TRAIN_SCRIPT_URL, TRAIN_SCRIPT)
txt = TRAIN_SCRIPT.read_text(encoding="utf-8")

assert "load_peft_weights_with_direct_fallback" in txt
assert "PEFT_MANUAL_LOAD_METHOD" in txt

print("OK: hf_job_train_v90.py corrigido no runtime.")


OK: hf_job_train_v90.py corrigido no runtime.


In [11]:
import os, shutil
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)
os.environ['UPLOAD_TO_HF'] = '0'
os.environ['MODEL_NAME'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
os.environ['DATA_FILE'] = '/content/kg1_v198/data/v198/v198_micro_train.strict.jsonl'
os.environ['VAL_FILE'] = '/content/kg1_v198/data/v198/v198_micro_val.strict.jsonl'
os.environ['INIT_ADAPTER_DIR'] = str(INIT_ADAPTER)
os.environ['INIT_ADAPTER_LOAD_MODE'] = 'manual'
os.environ['OUTPUT_DIR'] = str(OUT)
os.environ['V198_OUT'] = str(OUT)
os.environ['RUN_ID'] = 'v198-micro-distill-v197-gates'
os.environ['MAX_LENGTH'] = '2048'
os.environ['BATCH_SIZE'] = '16'
os.environ['MICRO_BATCH_SIZE'] = '1'
os.environ['GRADIENT_CHECKPOINTING'] = '1'
os.environ['MAX_STEPS'] = '45'
os.environ['SAVE_EVERY_STEPS'] = '15'
os.environ['EVAL_EVERY_STEPS'] = '15'
os.environ['EVAL_MAX_EXAMPLES'] = '240'
os.environ['LEARNING_RATE'] = '1e-5'
os.environ['FINAL_LEARNING_RATE'] = '3e-6'
os.environ['EXPECTED_TRAIN_SHA256'] = '6d2742616300818eb50c54d36019551b24f5b71c607a2b28feda7461a709def0'
os.environ['EXPECTED_VAL_SHA256'] = 'e59c907c6545e5e587097a64762e3e874508e8cd74d85d5c7c79354ebe56e73c'
os.environ['MIN_TRAIN_EXAMPLES'] = '1875'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES'] = '1600'
os.environ['MIN_VAL_EXAMPLES'] = '720'
os.environ['MIN_TOKENIZED_VAL_EXAMPLES'] = '700'
os.environ['TRAINABLE_LORA_MODULES'] = 'in_proj,out_proj,q_proj,k_proj,v_proj,o_proj'
os.environ['MAX_TRAINABLE_PARAM_RATIO'] = '0.035'
!python scripts/hf_job_train_v90.py


KG1 v90 category-solver remote training
Run ID: v198-micro-distill-v197-gates
Model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
Model revision: cbd3fa9f933d55ef16a84236559f4ee2a0526848
Train: felipesp1983/kg1-nemotron-training//content/kg1_v198/data/v198/v198_micro_train.strict.jsonl
Validation: felipesp1983/kg1-nemotron-training//content/kg1_v198/data/v198/v198_micro_val.strict.jsonl
Output repo: felipesp1983/kg1-nemotron-lora-v90-category-solver
Upload to HF: False
Require offset masks: True
Dry-run validate only: False
Initial adapter: /content/drive/MyDrive/KG1_NVIDIA_V195/output_v195/checkpoint-55
LoRA: r=32 alpha=32 dropout=0.0 target_modules=down_proj,in_proj,k_proj,lm_head,o_proj,out_proj,q_proj,up_proj,v_proj
Trainable LoRA module filter: in_proj,out_proj,q_proj,k_proj,v_proj,o_proj
Max trainable parameter ratio: 3.5000%
Length=2048 batch=16 micro_batch=1
LR: 1.00e-05 -> 3.00e-06
Epochs=1 max_steps=45

Loading tokenizer from nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16...
Using 

Convert the trained adapter to Kaggle layout. This does not submit to Kaggle.


In [12]:
!python scripts/kg1_convert_local_training_adapter_to_kaggle_zip.py \
  --source-adapter-dir "$V198_OUT/final_adapter" \
  --output-dir "$V198_OUT/kaggle_layout" \
  --run-id v198-micro-distill-v197-gates


{
  "decision": {
    "kaggle_layout_ready": true,
    "note": "Submission still requires local validation and explicit authorization."
  },
  "generated_at": "2026-05-03T21:21:32+00:00",
  "input": {
    "adapter_model_bytes": 4259063856,
    "adapter_model_sha256": "dd718b0d416fd9cd6ed928e90e185c131fee9d4cb956f57e59b7d00c3266dafa"
  },
  "output": {
    "adapter_model_bytes": 4259028080,
    "adapter_model_sha256": "bca1ac0c262cdda32018b25434b544958b0444027e09b20e2f42b9e924c5b693",
    "already_kaggle_count": 3,
    "converted_tensor_count": 12011,
    "renamed_backbone_to_model_count": 12008,
    "tensor_count": 12011,
    "unchanged_count": 0,
    "unexpected_unchanged_prefix_sample": [],
    "zip_bytes": 3818230586,
    "zip_sha256": "4923a8b3411577dc13da683eb1fc947cc4deed1d89d836ccfa2520acd2df4176"
  },
  "output_adapter_dir": "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/kaggle_layout/adapter",
  "run_id": "v198-micro-distill-v197-gates",
  "source_adapter_dir": "/content/

In [13]:
import pathlib, urllib.request

POSTTRAIN_SCRIPT = pathlib.Path('/content/kg1_v198/scripts/kg1_v198_posttrain_gate.py')
POSTTRAIN_SCRIPT_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/cee9825b0edd6ea2e829c94bdd7b1ff9410b30f3/scripts/kg1_v198_posttrain_gate.py'

if not POSTTRAIN_SCRIPT.exists():
    print('Downloading V198 posttrain gate script...')
    urllib.request.urlretrieve(POSTTRAIN_SCRIPT_URL, POSTTRAIN_SCRIPT)

!python /content/kg1_v198/scripts/kg1_v198_posttrain_gate.py \
  --root /content/kg1_v198 \
  --output-root /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198 \
  --fail-on-block


{
  "decision": {
    "kaggle_layout_ready": true,
    "note": "Submission still requires local validation and explicit authorization."
  },
  "generated_at": "2026-05-03T21:31:06+00:00",
  "input": {
    "adapter_model_bytes": 4259063856,
    "adapter_model_sha256": "dd718b0d416fd9cd6ed928e90e185c131fee9d4cb956f57e59b7d00c3266dafa"
  },
  "output": {
    "adapter_model_bytes": 4259028080,
    "adapter_model_sha256": "bca1ac0c262cdda32018b25434b544958b0444027e09b20e2f42b9e924c5b693",
    "already_kaggle_count": 3,
    "converted_tensor_count": 12011,
    "renamed_backbone_to_model_count": 12008,
    "tensor_count": 12011,
    "unchanged_count": 0,
    "unexpected_unchanged_prefix_sample": [],
    "zip_bytes": 3818230586,
    "zip_sha256": "52c585c7f075a1a9735d23c16905e535d1ebbf51246b03a50ac3d07c3768a3a9"
  },
  "output_adapter_dir": "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/adapter",
  "run_id": "v198-micro-distill-final",
  "source_adapter_dir": "

In [19]:
import pathlib, urllib.request

ROOT = pathlib.Path("/content/kg1_v198")
SCRIPTS = ROOT / "scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/e80dab6233f896e0542bfded2ea20382ff68297f/scripts"

for name in ["nemotron_submission_preflight.py", "kg1_submission_gate.py"]:
    url = f"{BASE}/{name}"
    dst = SCRIPTS / name
    print("downloading", url)
    urllib.request.urlretrieve(url, dst)
    assert dst.exists(), f"missing {dst}"

print("scripts ready")


downloading https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/e80dab6233f896e0542bfded2ea20382ff68297f/scripts/nemotron_submission_preflight.py


HTTPError: HTTP Error 404: Not Found

In [16]:
FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"

!python /content/kg1_v198/scripts/nemotron_submission_preflight.py \
  --adapter-zip "$FINAL_ZIP" \
  --output-json /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_preflight.json \
  --fail-on-block


python3: can't open file '/content/kg1_v198/scripts/nemotron_submission_preflight.py': [Errno 2] No such file or directory


In [17]:
CKPT30_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/checkpoint30/zip/v198-micro-distill-checkpoint30_adapter_only.zip"

!python /content/kg1_v198/scripts/nemotron_submission_preflight.py \
  --adapter-zip "$CKPT30_ZIP" \
  --output-json /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/checkpoint30_preflight.json \
  --fail-on-block


python3: can't open file '/content/kg1_v198/scripts/nemotron_submission_preflight.py': [Errno 2] No such file or directory


In [20]:
import pathlib, urllib.request

ROOT = pathlib.Path("/content/kg1_v198")
SCRIPTS = ROOT / "scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/8a1c0b934acde30c81237d33a74a18d11f6d5141/scripts"

for name in ["nemotron_submission_preflight.py", "kg1_submission_gate.py"]:
    url = f"{BASE}/{name}"
    dst = SCRIPTS / name
    print("downloading", url)
    urllib.request.urlretrieve(url, dst)
    assert dst.exists(), f"missing {dst}"

print("scripts ready")


downloading https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/8a1c0b934acde30c81237d33a74a18d11f6d5141/scripts/nemotron_submission_preflight.py
downloading https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/8a1c0b934acde30c81237d33a74a18d11f6d5141/scripts/kg1_submission_gate.py
scripts ready


In [21]:
FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"

!python /content/kg1_v198/scripts/nemotron_submission_preflight.py \
  --adapter-zip "$FINAL_ZIP" \
  --output-json /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_preflight.json \
  --fail-on-block


production_ready: True
report: /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_preflight.json


In [22]:
import pathlib, urllib.request

ROOT = pathlib.Path("/content/kg1_v198")
SCRIPTS = ROOT / "scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)

URL = "https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/b2209b4daefb85e9d3c9e4dc5e26b3ba54dbcbd2/scripts/kg1_v198_final_submit_doublecheck.py"
DST = SCRIPTS / "kg1_v198_final_submit_doublecheck.py"

print("downloading", URL)
urllib.request.urlretrieve(URL, DST)
assert DST.exists(), f"missing {DST}"

FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"

!python /content/kg1_v198/scripts/kg1_v198_final_submit_doublecheck.py \
  --candidate-zip "$FINAL_ZIP" \
  --expected-label final \
  --output-json /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_submit_doublecheck.json \
  --fail-on-block


downloading https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/b2209b4daefb85e9d3c9e4dc5e26b3ba54dbcbd2/scripts/kg1_v198_final_submit_doublecheck.py

=== V198 FINAL SUBMIT DOUBLECHECK ===
candidate_zip: /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip
zip_sha256: 52c585c7f075a1a9735d23c16905e535d1ebbf51246b03a50ac3d07c3768a3a9
adapter_model_sha256: bca1ac0c262cdda32018b25434b544958b0444027e09b20e2f42b9e924c5b693
submit_ready: True
report: /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_submit_doublecheck.json


In [37]:
!kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f /content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip \
  -m "V198 micro distill final eval_loss 0.9013 preflight OK"


401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload


In [24]:
FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"

!kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f "$FINAL_ZIP" \
  -m "V198 micro distill final eval_loss 0.9013 gates OK sha 52c585c7"


You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [32]:
FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"
DOUBLECHECK = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_submit_doublecheck.json"
REPORT = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/v198_safe_submit_report.json"

!python /content/kg1_v198/scripts/kg1_v198_safe_kaggle_submit.py \
  --candidate-zip "$FINAL_ZIP" \
  --doublecheck-json "$DOUBLECHECK" \
  --expected-sha256 52c585c7f075a1a9735d23c16905e535d1ebbf51246b03a50ac3d07c3768a3a9 \
  --message "V198 micro distill final eval_loss 0.9013 all gates OK sha 52c585c7" \
  --output-json "$REPORT" \
  --poll-seconds 60


python3: can't open file '/content/kg1_v198/scripts/kg1_v198_safe_kaggle_submit.py': [Errno 2] No such file or directory


In [39]:
import os, json, pathlib, shutil
from google.colab import files

kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

drive_kaggle = pathlib.Path("/content/drive/MyDrive/kaggle.json")

if drive_kaggle.exists():
    shutil.copy(drive_kaggle, kaggle_json)
else:
    print("Envie o arquivo kaggle.json da sua conta Kaggle")
    uploaded = files.upload()
    assert "kaggle.json" in uploaded, "kaggle.json nao enviado"
    kaggle_json.write_bytes(uploaded["kaggle.json"])

os.chmod(kaggle_json, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)

creds = json.loads(kaggle_json.read_text())
assert creds.get("username") and creds.get("key"), "kaggle.json invalido"

print("Kaggle auth OK:", creds["username"])


Kaggle auth OK: felipe1983


In [40]:
FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"

!kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f "$FINAL_ZIP" \
  -m "V198 micro distill final eval_loss 0.9013 gates OK sha 52c585c7"


401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload


In [41]:
import os, json, pathlib, subprocess
from google.colab import userdata

KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
KAGGLE_KEY = userdata.get("KAGGLE_KEY")

assert KAGGLE_USERNAME, "Secret KAGGLE_USERNAME ausente"
assert KAGGLE_KEY, "Secret KAGGLE_KEY ausente"

kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(json.dumps({
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_KEY,
}), encoding="utf-8")

os.chmod(kaggle_json, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

subprocess.run(["python", "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

print("Kaggle secrets auth file rebuilt for:", KAGGLE_USERNAME)


Kaggle secrets auth file rebuilt for: felipe1983


In [42]:
FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"

!kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f "$FINAL_ZIP" \
  -m "V198 micro distill final eval_loss 0.9013 gates OK sha 52c585c7"


100% 3.56G/3.56G [00:39<00:00, 96.6MB/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission


In [46]:
import os, json, pathlib, subprocess, sys, urllib.request
from google.colab import userdata

ROOT = pathlib.Path("/content/kg1_v198")
SCRIPTS = ROOT / "scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)

kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

username = userdata.get("KAGGLE_USERNAME")
key = userdata.get("KAGGLE_KEY")
assert username, "Secret KAGGLE_USERNAME ausente"
assert key, "Secret KAGGLE_KEY ausente"

kaggle_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
os.chmod(kaggle_json, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

SCRIPT = SCRIPTS / "kg1_v198_safe_kaggle_submit.py"
URL = "https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/cee234f4bac2abcff9f1452a440f0f7576e62eef/scripts/kg1_v198_safe_kaggle_submit.py"
urllib.request.urlretrieve(URL, SCRIPT)

FINAL_ZIP = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final/zip/v198-micro-distill-final_adapter_only.zip"
DOUBLECHECK = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_submit_doublecheck.json"
REPORT = "/content/drive/MyDrive/KG1_NVIDIA_V198/output_v198/posttrain_kaggle_gate/final_safe_submit_report_submission_name_fixed.json"

subprocess.run([
    sys.executable, str(SCRIPT),
    "--candidate-zip", FINAL_ZIP,
    "--doublecheck-json", DOUBLECHECK,
    "--expected-sha256", "52c585c7f075a1a9735d23c16905e535d1ebbf51246b03a50ac3d07c3768a3a9",
    "--message", "V198 micro distill final eval_loss 0.9013 gates OK sha 52c585c7",
    "--output-json", REPORT,
    "--poll-seconds", "60",
    "--no-raise-on-submit-error",
], check=False)

print(pathlib.Path(REPORT).read_text()[:12000])


{
  "competition": "nvidia-nemotron-model-reasoning-challenge",
  "credentials": {
    "path": "/root/.kaggle/kaggle.json",
    "source": "/root/.kaggle/kaggle.json",
    "username": "felipe1983"
  },
  "dry_run": false,
  "generated_at": "2026-05-03T23:17:41+00:00",
  "message": "V198 micro distill final eval_loss 0.9013 gates OK sha 52c585c7",
  "submissions_after": [
    {
      "date": "2026-05-03 23:18:23.073000",
      "description": "V198 micro distill final eval_loss 0.9013 gates OK sha 52c585c7",
      "private_score": null,
      "public_score": null,
      "ref": 52301667,
      "status": "SubmissionStatus.PENDING"
    },
    {
      "date": "2026-05-03 01:09:01.380000",
      "description": "v194 attention-only micro-merge aaitdads98p5 lineage1p5 keep lmhead experts sha49886191 gate-doublecheck-pass",
      "private_score": null,
      "public_score": null,
      "ref": 52275052,
      "status": "SubmissionStatus.COMPLETE"
    },
    {
      "date": "2026-05-02 23:40:04.950

In [ ]:
import os, json, pathlib, time
from google.colab import userdata
from kaggle.api.kaggle_api_extended import KaggleApi

kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / "kaggle.json").write_text(json.dumps({
    "username": userdata.get("KAGGLE_USERNAME"),
    "key": userdata.get("KAGGLE_KEY"),
}), encoding="utf-8")
os.chmod(kaggle_dir / "kaggle.json", 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)

COMP = "nvidia-nemotron-model-reasoning-challenge"
TARGET_REF = 52301667

api = KaggleApi()
api.authenticate()

for i in range(60):
    subs = api.competition_submissions(COMP)
    target = next((s for s in subs if int(getattr(s, "ref", -1)) == TARGET_REF), None)

    if target is None:
        print("Ref ainda não apareceu:", TARGET_REF)
    else:
        print({
            "ref": target.ref,
            "date": str(target.date),
            "status": str(target.status),
            "public_score": getattr(target, "publicScore", None),
            "private_score": getattr(target, "privateScore", None),
            "description": getattr(target, "description", None),
        })

        if "PENDING" not in str(target.status):
            break

    time.sleep(60)
